# SMOTE influence on metrics

This notebook evaluates the influence of using SMOTE and SMOTENC, both with default parameters, on the resulting metrics. For SMOTENC, the features "wettability" and "inclination" were provided.

# Prerequisites

In [1]:
import os
from pathlib import Path
import seaborn as sns
import joblib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

## Load Data

In [2]:
df_metrics = pd.read_excel(
    Path(
        '../results/metrics_modelling4.xlsx'
    ),
)
display(df_metrics.head())
df_metrics.info()

,dataset,target,model,params,holdout_test_accuracy,holdout_test_precision,holdout_test_recall,holdout_test_f1,holdout_test_roc_auc,holdout_test_f1_macro,...,cv_train_recall_class_0_mean,cv_train_recall_class_0_median,cv_train_recall_class_0_std,cv_train_recall_mean,cv_train_recall_median,cv_train_recall_std,cv_train_roc_auc,cv_train_roc_auc_mean,cv_train_roc_auc_median,cv_train_roc_auc_std
0,df_dimless,splashing,Logit_splashing_default,{'estimator': StatsModelsEstimator(model_class...,0.800000,0.823529,0.875000,0.848485,0.874228,0.777184,...,0.784073,0.778761,0.017802,0.845821,0.844660,0.010156,"0.8959421541118067, 0.9114676936243047, 0.9020...",0.905980,0.907788,0.006070
1,df_dimless,no_fragmentation,Logit_no_fragmentation_default,{'estimator': StatsModelsEstimator(model_class...,0.906667,0.809524,0.850000,0.829268,0.951818,0.882524,...,0.916361,0.914530,0.012056,0.760984,0.752941,0.021087,"0.9395604395604396, 0.9486676721970839, 0.9361...",0.940673,0.941277,0.005873
2,df_dimless,splashing,LogisticRegression_splashing_default,{'estimator': LogisticRegression(fit_intercept...,0.800000,0.823529,0.875000,0.848485,0.879630,0.777184,...,0.803036,0.796460,0.021463,0.838192,0.834951,0.009285,"0.8971077055903303, 0.9126658108686351, 0.9032...",0.907306,0.909206,0.006459
3,df_dimless,no_fragmentation,LogisticRegression_no_fragmentation_default,{'estimator': LogisticRegression(fit_intercept...,0.826667,0.629630,0.850000,0.723404,0.946364,0.798595,...,0.843712,0.846154,0.010179,0.878792,0.870588,0.014996,"0.9391534391534391, 0.949270990447461, 0.93559...",0.940457,0.941679,0.005749
4,df_dimless,splashing,KNeighborsClassifier_splashing_default,"{'estimator': KNeighborsClassifier(), 'source_...",0.853333,0.862745,0.916667,0.888889,0.880787,0.836601,...,0.845942,0.849558,0.017741,0.916667,0.917476,0.010215,"0.9671918843082237, 0.9732563115104835, 0.9687...",0.968197,0.967738,0.002584


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Columns: 112 entries, dataset to cv_train_roc_auc_std
dtypes: float64(86), object(26)
memory usage: 47.4+ KB


## Apply SMOTE and pure model groups

In [3]:
df_metrics['model_class'] = df_metrics['model'].apply(
    lambda x: x.split('_')[0]
)

df_metrics['model_group'] = df_metrics['model'].apply(
    lambda x: x.split('_')[-1]
)

def define_SMOTE_type(model):
    if 'smotenc' in model:
        return 'SMOTENC'
    elif 'smote' in model:
        return 'SMOTE'
    else:
        return 'No'
        

df_metrics['SMOTE_type'] = df_metrics['model'].apply(
    define_SMOTE_type
)

df_metrics.head()

,dataset,target,model,params,holdout_test_accuracy,holdout_test_precision,holdout_test_recall,holdout_test_f1,holdout_test_roc_auc,holdout_test_f1_macro,...,cv_train_recall_mean,cv_train_recall_median,cv_train_recall_std,cv_train_roc_auc,cv_train_roc_auc_mean,cv_train_roc_auc_median,cv_train_roc_auc_std,model_class,model_group,SMOTE_type
0,df_dimless,splashing,Logit_splashing_default,{'estimator': StatsModelsEstimator(model_class...,0.800000,0.823529,0.875000,0.848485,0.874228,0.777184,...,0.845821,0.844660,0.010156,"0.8959421541118067, 0.9114676936243047, 0.9020...",0.905980,0.907788,0.006070,Logit,default,No
1,df_dimless,no_fragmentation,Logit_no_fragmentation_default,{'estimator': StatsModelsEstimator(model_class...,0.906667,0.809524,0.850000,0.829268,0.951818,0.882524,...,0.760984,0.752941,0.021087,"0.9395604395604396, 0.9486676721970839, 0.9361...",0.940673,0.941277,0.005873,Logit,default,No
2,df_dimless,splashing,LogisticRegression_splashing_default,{'estimator': LogisticRegression(fit_intercept...,0.800000,0.823529,0.875000,0.848485,0.879630,0.777184,...,0.838192,0.834951,0.009285,"0.8971077055903303, 0.9126658108686351, 0.9032...",0.907306,0.909206,0.006459,LogisticRegression,default,No
3,df_dimless,no_fragmentation,LogisticRegression_no_fragmentation_default,{'estimator': LogisticRegression(fit_intercept...,0.826667,0.629630,0.850000,0.723404,0.946364,0.798595,...,0.878792,0.870588,0.014996,"0.9391534391534391, 0.949270990447461, 0.93559...",0.940457,0.941679,0.005749,LogisticRegression,default,No
4,df_dimless,splashing,KNeighborsClassifier_splashing_default,"{'estimator': KNeighborsClassifier(), 'source_...",0.853333,0.862745,0.916667,0.888889,0.880787,0.836601,...,0.916667,0.917476,0.010215,"0.9671918843082237, 0.9732563115104835, 0.9687...",0.968197,0.967738,0.002584,KNeighborsClassifier,default,No


## Get best SMOTE types

In [4]:
metric_list_holdout = ['holdout_test_f1_macro', 'holdout_test_roc_auc', 'holdout_test_accuracy_balanced',]

metric_list_cv = ['cv_test_f1_macro_median', 'cv_test_roc_auc_median', 'cv_test_accuracy_balanced_median',]

metrics_by_target = {
    target: df_metrics[df_metrics['target'] == target]
    for target in df_metrics['target'].unique()
}

for metric_list in [metric_list_holdout, metric_list_cv]:

    smote_by_target = {}

    for target in metrics_by_target:
        print(target.upper())
        print(metric_list)
        metrics = metrics_by_target[target].groupby(
            ['model_class', 'SMOTE_type']
        )[metric_list].median()

        metrics = metrics.sort_values(by=metric_list, ascending=False)

        result = metrics.reset_index().groupby('model_class')['SMOTE_type'].first()

        display(result)
        print('Best SMOTE type:', result.value_counts().idxmax())
        display(metrics)
        
        smote_by_target[target] = result

SPLASHING
['holdout_test_f1_macro', 'holdout_test_roc_auc', 'holdout_test_accuracy_balanced']


model_class
AdaBoostClassifier             No
CatBoostClassifier          SMOTE
KNeighborsClassifier           No
LGBMClassifier                 No
LogisticRegression             No
Logit                          No
RandomForestClassifier         No
SVC                         SMOTE
XGBClassifier             SMOTENC
Name: SMOTE_type, dtype: object

Best SMOTE type: No


holdout_test_f1_macro  \
model_class            SMOTE_type                          
CatBoostClassifier     SMOTE                    0.882261   
XGBClassifier          SMOTENC                  0.880000   
CatBoostClassifier     SMOTENC                  0.868703   
LGBMClassifier         No                       0.866310   
CatBoostClassifier     No                       0.863609   
XGBClassifier          SMOTE                    0.863609   
                       No                       0.863609   
LGBMClassifier         SMOTENC                  0.852826   
                       SMOTE                    0.852826   
RandomForestClassifier No                       0.850000   
                       SMOTE                    0.836601   
                       SMOTENC                  0.836601   
KNeighborsClassifier   No                       0.836601   
                       SMOTENC                  0.829027   
SVC                    SMOTE                    0.810348   
                       SMOTENC                  0.810348   
KNeighborsClassifier   SMOTE                    0.803223   
SVC                    No                       0.793956   
AdaBoostClassifier     No                       0.793956   
LogisticRegression     No                       0.777184   
Logit                  No                       0.777184   
LogisticRegression     SMOTE                    0.772036   
Logit                  SMOTENC                  0.764521   
                       SMOTE                    0.764521   
LogisticRegression     SMOTENC                  0.755981   
AdaBoostClassifier     SMOTE                    0.751994   
                       SMOTENC                  0.751994   

                                   holdout_test_roc_auc  \
model_class            SMOTE_type                         
CatBoostClassifier     SMOTE                   0.950617   
XGBClassifier          SMOTENC                 0.932485   
CatBoostClassifier     SMOTENC                 0.948302   
LGBMClassifier         No                      0.951775   
CatBoostClassifier     No                      0.955247   
XGBClassifier          SMOTE                   0.942515   
                       No                      0.940972   
LGBMClassifier         SMOTENC                 0.957562   
                       SMOTE                   0.949074   
RandomForestClassifier No                      0.941744   
                       SMOTE                   0.939815   
                       SMOTENC                 0.937886   
KNeighborsClassifier   No                      0.880787   
                       SMOTENC                 0.887346   
SVC                    SMOTE                   0.885031   
                       SMOTENC                 0.877315   
KNeighborsClassifier   SMOTE                   0.887346   
SVC                    No                      0.911265   
AdaBoostClassifier     No                      0.881559   
LogisticRegression     No                      0.879630   
Logit                  No                      0.874228   
LogisticRegression     SMOTE                   0.879630   
Logit                  SMOTENC                 0.871142   
                       SMOTE                   0.868827   
LogisticRegression     SMOTENC                 0.878858   
AdaBoostClassifier     SMOTE                   0.839892   
                       SMOTENC                 0.830633   

                                   holdout_test_accuracy_balanced  
model_class            SMOTE_type                                  
CatBoostClassifier     SMOTE                             0.876157  
XGBClassifier          SMOTENC                           0.868056  
CatBoostClassifier     SMOTENC                           0.865741  
LGBMClassifier         No                                0.857639  
CatBoostClassifier     No                                0.849537  
XGBClassifier          SMOTE                             0.849537  
                       No             

NO_FRAGMENTATION
['holdout_test_f1_macro', 'holdout_test_roc_auc', 'holdout_test_accuracy_balanced']


...

cv_test_f1_macro_median  \
model_class            SMOTE_type                            
RandomForestClassifier SMOTE                      0.898584   
LGBMClassifier         SMOTENC                    0.898584   
CatBoostClassifier     No                         0.896201   
                       SMOTE                      0.881696   
                       SMOTENC                    0.876935   
XGBClassifier          SMOTE                      0.876935   
                       SMOTENC                    0.876935   
LGBMClassifier         SMOTE                      0.876935   
                       No                         0.873810   
XGBClassifier          No                         0.873810   
RandomForestClassifier No                         0.873810   
                       SMOTENC                    0.858018   
SVC                    SMOTE                      0.844575   
KNeighborsClassifier   SMOTENC                    0.844575   
AdaBoostClassifier     SMOTE                      0.839394   
SVC                    SMOTENC                    0.826230   
KNeighborsClassifier   SMOTE                      0.826230   
SVC                    No                         0.821013   
KNeighborsClassifier   No                         0.821013   
AdaBoostClassifier     No                         0.817451   
Logit                  No                         0.817451   
AdaBoostClassifier     SMOTENC                    0.799242   
Logit                  SMOTENC                    0.799242   
LogisticRegression     SMOTENC                    0.799242   
                       SMOTE                      0.799242   
                       No                         0.799242   
Logit                  SMOTE                      0.787614   

                                   cv_test_roc_auc_median  \
model_class            SMOTE_type                           
RandomForestClassifier SMOTE                     0.957430   
LGBMClassifier         SMOTENC                   0.948916   
CatBoostClassifier     No                        0.953560   
                       SMOTE                     0.950464   
                       SMOTENC                   0.956656   
XGBClassifier          SMOTE                     0.953560   
                       SMOTENC                   0.946032   
LGBMClassifier         SMOTE                     0.945820   
                       No                        0.945820   
XGBClassifier          No                        0.942724   
RandomForestClassifier No                        0.933437   
                       SMOTENC                   0.945046   
SVC                    SMOTE                     0.922601   
KNeighborsClassifier   SMOTENC                   0.917957   
AdaBoostClassifier     SMOTE                     0.907121   
SVC                    SMOTENC                   0.933437   
KNeighborsClassifier   SMOTE                     0.917957   
SVC                    No                        0.928793   
KNeighborsClassifier   No                        0.923308   
AdaBoostClassifier     No                        0.900929   
Logit                  No                        0.869969   
AdaBoostClassifier     SMOTENC                   0.902477   
Logit                  SMOTENC                   0.877709   
LogisticRegression     SMOTENC                   0.873065   
                       SMOTE                     0.873065   
                       No                        0.871517   
Logit                  SMOTE                     0.876161   

                                   cv_test_accuracy_balanced_median  
model_class            SMOTE_type                                    
RandomForestClassifier SMOTE                               0.903251  
LGBMClassifier         SMOTENC                             0.903251  
CatBoostClassifier     No                                  0.891641  
                       SMOTE                               0.900794  
                       SMOTENC                

NO_FRAGMENTATION
['cv_test_f1_macro_median', 'cv_test_roc_auc_median', 'cv_test_accuracy_balanced_median']


model_class
AdaBoostClassifier          SMOTE
CatBoostClassifier          SMOTE
KNeighborsClassifier      SMOTENC
LGBMClassifier                 No
LogisticRegression             No
Logit                       SMOTE
RandomForestClassifier      SMOTE
SVC                       SMOTENC
XGBClassifier                  No
Name: SMOTE_type, dtype: object

Best SMOTE type: SMOTE


cv_test_f1_macro_median  \
model_class            SMOTE_type                            
CatBoostClassifier     SMOTE                      0.907692   
                       No                         0.898077   
LGBMClassifier         No                         0.898077   
                       SMOTE                      0.886022   
                       SMOTENC                    0.886022   
CatBoostClassifier     SMOTENC                    0.881326   
SVC                    SMOTENC                    0.881326   
KNeighborsClassifier   SMOTENC                    0.881326   
XGBClassifier          No                         0.875762   
                       SMOTENC                    0.875762   
                       SMOTE                      0.875762   
RandomForestClassifier SMOTE                      0.875762   
                       SMOTENC                    0.875762   
                       No                         0.869136   
LogisticRegression     No                         0.854396   
SVC                    SMOTE                      0.850704   
KNeighborsClassifier   SMOTE                      0.850704   
AdaBoostClassifier     SMOTE                      0.850704   
KNeighborsClassifier   No                         0.835007   
Logit                  SMOTE                      0.828299   
AdaBoostClassifier     SMOTENC                    0.826067   
                       No                         0.826067   
Logit                  No                         0.814035   
                       SMOTENC                    0.814035   
LogisticRegression     SMOTE                      0.813161   
                       SMOTENC                    0.813161   
SVC                    No                         0.796154   

                                   cv_test_roc_auc_median  \
model_class            SMOTE_type                           
CatBoostClassifier     SMOTE                     0.974359   
                       No                        0.979487   
LGBMClassifier         No                        0.958974   
                       SMOTE                     0.976068   
                       SMOTENC                   0.974359   
CatBoostClassifier     SMOTENC                   0.972650   
SVC                    SMOTENC                   0.963370   
KNeighborsClassifier   SMOTENC                   0.927656   
XGBClassifier          No                        0.979487   
                       SMOTENC                   0.967033   
                       SMOTE                     0.967033   
RandomForestClassifier SMOTE                     0.956410   
                       SMOTENC                   0.954701   
                       No                        0.952137   
LogisticRegression     No                        0.945055   
SVC                    SMOTE                     0.957875   
KNeighborsClassifier   SMOTE                     0.952381   
AdaBoostClassifier     SMOTE                     0.934066   
KNeighborsClassifier   No                        0.932234   
Logit                  SMOTE                     0.935897   
AdaBoostClassifier     SMOTENC                   0.930403   
                       No                        0.912088   
Logit                  No                        0.932234   
                       SMOTENC                   0.930403   
LogisticRegression     SMOTE                     0.943223   
                       SMOTENC                   0.939560   
SVC                    No                        0.943223   

                                   cv_test_accuracy_balanced_median  
model_class            SMOTE_type                                    
CatBoostClassifier     SMOTE                               0.907692  
                       No                                  0.880037  
LGBMClassifier         No                                  0.890110  
                       SMOTE                               0.902930  
                       SMOTENC                

# **CONCLUSIONS:**
- For Logit, there is sufficient justification for not using SMOTE[NC], even without considering statistical significance.
- For CV metrics, the best choice (based on the number of models with the best metric scores) was **SMOTE**.
- For holdout, the absence of SMOTE/SMOTENC is preferable.

Thus, for all models except the non-optimizable Logit, SMOTE is used.